<!-- colab-badge -->
[![Open in Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/RemusTeodorescu/DL-for-Engineers-Public-Course/blob/main/exercises/Ex08.1-stationary-heat/Ex08.1_01_manufactured_light.ipynb)

*Open this notebook in Google Colab. Its first code cell fetches the set's library files from the public course repository, so nothing needs uploading.*
*This is the **light** version: every TODO is written out, and you only replace the `...` marked lines with what the comment beside them says.*


<!-- course-header v3 -->

**Deep Learning for Engineering** · MSc, Aalborg University · 2026

Developed by **Remus Teodorescu** (ret@et.aau.dk), with support from Research Assistant **Noman Khan** (nomank@energy.aau.dk).

*Reference texts — read for the theory. Used as an **inspirational source** for this course, not as a source of its code:*

- Liu, *PINN with Python*, 2025.
- Prince, *Understanding Deep Learning*, MIT Press 2023.

Every notebook in this course has been **written and rewritten by the authors named above**. The code, the problems, the data and the exposition are **original to this course** and are not derived from any publisher's code listings or companion notebooks. Where notation matches a textbook's it is the standard notation of the field, and where an idea is a named author's it is cited as theirs in the text.

See `docs/PROVENANCE.md` for what each reference is cited for, set by set.

---

# Ex_08.1 · Notebook 01 — Verify on a Manufactured Solution

**Paired with L8.1 · Stationary Heat Transfer**

Before any geometry, confirm the formulation on the unit square with a known
answer:

$$T = \sin(\pi x)\sin(\pi y) \qquad\Rightarrow\qquad
\frac{Q}{k} = 2\pi^2 \sin(\pi x)\sin(\pi y)$$

This is the manufactured field L8.1 slide 18 uses, and the point of slide 18 is
the point of this notebook: **a bug found on the unit square costs minutes; the
same bug found on the plate with the hole costs an afternoon.**

The boundary condition here is $T = 0$ on all four edges of the unit square, so
it can be hard-enforced by the polynomial lift $x(1-x)\,y(1-y)$ — no penalty,
no weight, no wall error to report.

---

## 0 · Setup

In [ ]:
# files-cell v1 ----------------------------------------------------------
# This set's library files must sit beside the notebook. Locally they
# already do. On Google Colab, where a notebook opens on its own, they are
# fetched from the public course repository. Run this cell first.
import os, urllib.request
FILES = ['course_core.py', 'pinn_core.py', 'problem.py']
URL = "https://raw.githubusercontent.com/RemusTeodorescu/DL-for-Engineers-Public-Course/main/exercises/Ex08.1-stationary-heat/"
for f in FILES:
    if not os.path.exists(f):
        urllib.request.urlretrieve(URL + f, f)
        print("fetched", f)
print("files ready:", ", ".join(FILES))


In [ ]:
# --- setup: every Part 2 notebook opens with this cell ------------------
# Needs course_core.py, pinn_core.py and problem.py beside this notebook.
# On Colab the files cell above fetched them from the public course repository.
import os
for f in ("course_core.py", "pinn_core.py", "problem.py"):
    assert os.path.exists(f), f"{f} is missing - run the files cell above first"

from pinn_core import *                                  # noqa: F401,F403
import problem as pb
import numpy as np, torch, matplotlib.pyplot as plt

set_seed(88)
print("device:", DEVICE, " dtype:", torch.get_default_dtype())

## 1 · The network and the collocation set

`describe` prints the network's size and the ratio of collocation points to
trainable parameters. Read that ratio, and read the note under the cell — it is
**not** the sampling condition the L8.1 deck quotes.

In [ ]:
N_COLL = 320

set_seed(88)
model = MLP(n_in=2, n_hidden=40, n_layers=2)
describe(model, N_COLL)

xy_f = to_tensor(interior_points(N_COLL, pb.DOMAIN, method="lhs", seed=SEED),
                 requires_grad=True)
print("collocation tensor:", tuple(xy_f.shape),
      " requires_grad:", xy_f.requires_grad)

**What you should see.** A 2 → 40 × 2 → 1 network, `1801` trainable
parameters, and `(320, 2)` with `requires_grad: True`.

**Two different sizing rules, and they disagree here.** L8.1 slide 12 quotes
Liu's pseudo-dimension $P^* = (p+1) + (N_n+1)N_L$, which for this network is
$3 + 41 \times 2 = 85$, and the condition $N_{\text{sample}} \ge P^*$ — which
320 satisfies almost four times over. `describe` reports something else: 320
collocation points against 1801 parameters, which it prints as `0.2 x
parameters`.

Both numbers are honest and they measure different things. $P^*$ counts
*neurons*; the ratio counts *weights*. Keep the question in mind while you read
the error at the end of this notebook, and note what makes it survivable here:
the trial solution is hard-enforced, so a large part of the function space —
everything that is non-zero on the boundary — has already been removed before
training starts.

---

## 2 · The residual and the hard-enforced trial solution

### TODO 1 — residual and hard-enforced trial solution

In [ ]:
# TODO 1 --- trial solution, residual, loss -----------------------------------------------------------
# Three `...` to replace:
#   line 1  ->  x * (1 - x) * y * (1 - y) * model(xy)                     zero on all four edges of the unit square
#   line 2  ->  d2(T, xy, 0) + d2(T, xy, 1) + pb.source_manufactured(xy)   T_xx + T_yy + q
#   line 3  ->  mse(residual(model, xy_f))                                  no arguments: closes over model and xy_f
def trial(model, xy):
    x, y = xy[:, 0:1], xy[:, 1:2]
    return ...                                    # <- x * (1 - x) * y * (1 - y) * model(xy)

def residual(model, xy):
    T = trial(model, xy)
    return ...                                    # <- d2(T, xy, 0) + d2(T, xy, 1) + pb.source_manufactured(xy)

def loss_fn():
    return ...                                    # <- mse(residual(model, xy_f))
# ------------------------------------------------------------------------------

## 3 · Train

Adam to get close, L-BFGS to finish.

`train_two_stage` runs `lbfgs_steps` **outer** L-BFGS steps, each of which does
up to 20 inner iterations behind a strong-Wolfe line search. It is not the same
parameter as the old core's `lbfgs_epochs`, which was the inner count for a
single step — 150 here is a longer polish than the number suggests.

In [ ]:
history = train_two_stage(model, loss_fn,
                          adam_steps=2000, lbfgs_steps=150, lr=1e-3)

plot_curves(history, title="manufactured verification, hard BC")
plt.show()

## 4 · Score it against the answer you already knew

Two currencies, as always: a dimensionless norm for comparing across problems,
and the worst single point for the thing an engineer is certified on.

In [ ]:
X, Y, pts = grid_points(201, 201, pb.DOMAIN)
with torch.no_grad():
    T = to_numpy(trial(model, to_tensor(pts))).reshape(X.shape)
T_ref = pb.exact_manufactured(X, Y)

rel = relative_l2(T, T_ref)
mx = max_abs_error(T, T_ref)
print(f"    rel_L2 = {rel:.3e}")
print(f"   max_abs = {mx:.3e}")

edge = np.concatenate([np.abs(T - T_ref)[0, :], np.abs(T - T_ref)[-1, :],
                       np.abs(T - T_ref)[:, 0], np.abs(T - T_ref)[:, -1]])
print(f"  boundary = {edge.max():.3e}     <- hard-enforced, so ~1e-16")

fig, axes = plt.subplots(1, 2, figsize=(10.0, 4.2))
c0 = axes[0].contourf(X, Y, T, 50, cmap="magma")
axes[0].set_title("$\\hat{T}$ — the model"); axes[0].set_aspect("equal")
plt.colorbar(c0, ax=axes[0])
c1 = axes[1].contourf(X, Y, T - T_ref, 50, cmap="coolwarm")
axes[1].set_title("signed error, $\\hat{T} - T$"); axes[1].set_aspect("equal")
plt.colorbar(c1, ax=axes[1])
plt.tight_layout(); plt.show()

**Check.** Relative L2 should land near the hard-BC figure quoted in
L7.1, ~1e-04, and the boundary error should be at machine precision rather than
merely small — that is what "hard" means.

Note that L7.1's number was measured on a different manufactured field; the
order of magnitude is the comparison, not the digits.

---

## 5 · Save

In [ ]:
os.makedirs("Ex08.1_outputs", exist_ok=True)
path = os.path.join("Ex08.1_outputs", "nb01_manufactured.npz")
np.savez(path,
         rel_L2=rel, max_abs=mx, boundary=edge.max(), T=T,
         adam=history["adam"], lbfgs=history["lbfgs"])
torch.save(model.state_dict(),
           os.path.join("Ex08.1_outputs", "nb01_manufactured.pt"))
print("wrote", path)

## 6 · Before you move on

1. The boundary error is at machine precision and nothing in the loss asked for
   it. Say where it came from.
2. `loss_fn` has one term. Name the term that would have appeared if the
   boundary condition had been soft, and the number you would then have had to
   choose.
3. You now have a verified residual, a verified trial-solution pattern and a
   verified training loop. Which of the three does notebook 02 change?

Next: **notebook 02**, where the geometry stops being a rectangle.